# 01 — Feature Engineering
## Stationarity, Differencing, Lags & Rolling Statistics

### Purpose
Test stationarity of all indicators using ADF and KPSS tests.
First-difference non-stationary columns. Build lag features (t-1).
Compute rolling mean and standard deviation (3yr, 5yr).

### Input
- `../data/01_panel_cleaned.csv`

### Output
- `../data/02_panel_features.csv`

### Run after → Run before
`00_data_preparation.ipynb` → `02_instability_index_eda.ipynb`

In [19]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df_panel = pd.read_csv("data/01_panel_cleaned.csv")
print(f"Loaded: {df_panel.shape} | Countries: {df_panel['COUNTRY'].nunique()}")
df_panel.head()

Loaded: (5507, 23) | Countries: 175


,COUNTRY,YEAR,Inflation,Current_Account,Expenditure,Exports,Investment,Debt,GDP_Growth,Savings,...,Inflation_imputed,Current_Account_imputed,Expenditure_imputed,Exports_imputed,Imports_imputed,Savings_imputed,Investment_imputed,Debt_imputed,Fiscal_Balance_imputed,Revenue_imputed
0,"Afghanistan, Islamic Republic of",2003,35.663,29.616,11.927,49.575,30.102,270.602,8.692,59.718,...,False,False,False,False,False,False,False,False,False,False
1,"Afghanistan, Islamic Republic of",2004,16.358,37.216,15.069,-8.863,35.354,244.967,0.671,72.570,...,False,False,False,False,False,False,False,False,False,False
2,"Afghanistan, Islamic Republic of",2005,10.569,30.226,15.651,40.082,37.048,206.356,11.830,67.274,...,False,False,False,False,False,False,False,False,False,False
3,"Afghanistan, Islamic Republic of",2006,6.785,20.844,18.262,-5.943,29.489,22.985,5.361,50.333,...,False,False,False,False,False,False,False,False,False,False
4,"Afghanistan, Islamic Republic of",2007,8.681,63.390,21.448,-12.758,55.852,20.137,13.340,119.243,...,False,False,False,False,False,False,False,False,False,False


In [20]:
# ── Stationarity tests (ADF + KPSS) ──────────────────────────
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings("ignore")

cols_to_test = [
    "GDP_Growth","Inflation","Debt","Fiscal_Balance",
    "Current_Account","Exports","Imports",
    "Revenue","Expenditure","Savings","Investment"
]

results = []
for col in cols_to_test:
    adf_stats, kpss_stats = [], []
    for country, grp in df_panel.groupby("COUNTRY"):
        series = grp[col].dropna()
        if len(series) < 8:
            continue
        try:
            adf_stats.append(adfuller(series, autolag="AIC")[1])
        except: pass
        try:
            kpss_stats.append(kpss(series, regression="c", nlags="auto")[1])
        except: pass
    results.append({
        "Column"         : col,
        "ADF_pct_reject" : np.mean(np.array(adf_stats) < 0.05) * 100,
        "KPSS_pct_reject": np.mean(np.array(kpss_stats) < 0.05) * 100,
        "ADF_median_p"   : np.median(adf_stats),
        "KPSS_median_p"  : np.median(kpss_stats)
    })

stationarity_df = pd.DataFrame(results)
print(stationarity_df.to_string(index=False))

         Column  ADF_pct_reject  KPSS_pct_reject  ADF_median_p  KPSS_median_p
     GDP_Growth       75.428571        17.142857      0.000530       0.100000
      Inflation       70.857143        23.428571      0.008546       0.100000
           Debt       19.428571        47.428571      0.402455       0.063681
 Fiscal_Balance       44.571429        29.714286      0.062974       0.100000
Current_Account       32.571429        25.714286      0.132563       0.100000
        Exports       82.857143        16.571429      0.000069       0.100000
        Imports       87.428571         9.142857      0.000047       0.100000
        Revenue       27.428571        50.285714      0.264647       0.049801
    Expenditure       25.142857        49.142857      0.243236       0.051588
        Savings       27.428571        44.571429      0.218111       0.066623
     Investment       28.000000        38.285714      0.125953       0.088768


In [21]:
# ── First-difference non-stationary columns ───────────────────
non_stationary_cols = ["Debt","Expenditure","Revenue","Savings","Investment"]

for col in non_stationary_cols:
    df_panel[f"{col}_diff"] = df_panel.groupby("COUNTRY")[col]        .transform(lambda x: x.diff())

# Verify stationarity after differencing
for col in non_stationary_cols:
    pvals = []
    for country, grp in df_panel.groupby("COUNTRY"):
        s = grp[f"{col}_diff"].dropna()
        if len(s) >= 8:
            try: pvals.append(adfuller(s, autolag="AIC")[1])
            except: pass
    pct = np.mean(np.array(pvals) < 0.05) * 100
    print(f"{col}_diff → {pct:.1f}% stationary after differencing")

Debt_diff → 81.7% stationary after differencing
Expenditure_diff → 89.7% stationary after differencing
Revenue_diff → 88.6% stationary after differencing
Savings_diff → 86.9% stationary after differencing
Investment_diff → 89.1% stationary after differencing


In [22]:
# ── Rolling statistics (3yr and 5yr) ─────────────────────────
roll_cols = [
    "Inflation","GDP_Growth","Exports","Imports",
    "Fiscal_Balance","Current_Account",
    "Debt","Expenditure","Revenue","Savings","Investment"
]

for col in roll_cols:
    for w in [3, 5]:
        df_panel[f"{col}_rollmean{w}"] = df_panel.groupby("COUNTRY")[col]            .transform(lambda x: x.shift(1).rolling(w, min_periods=2).mean())
        df_panel[f"{col}_rollstd{w}"] = df_panel.groupby("COUNTRY")[col]            .transform(lambda x: x.shift(1).rolling(w, min_periods=2).std())

print(f"Rolling stats added. Total columns: {df_panel.shape[1]}")

Rolling stats added. Total columns: 72


In [23]:
# ============================================================
# Additional volatility features
# Use lagged rolling volatility only
# ============================================================

volatility_base_cols = [
    "GDP_Growth",
    "Inflation",
    "Exports",
    "Imports",
    "Fiscal_Balance",
    "Current_Account",
    "Debt",
    "Revenue",
    "Expenditure",
    "Savings",
    "Investment",
]

df_panel = df_panel.sort_values(["COUNTRY", "YEAR"]).copy()

for col in volatility_base_cols:
    if col in df_panel.columns:
        df_panel[f"{col}_rollstd3"] = (
            df_panel.groupby("COUNTRY")[col]
            .transform(
                lambda x: x.shift(1).rolling(
                    window=3,
                    min_periods=2,
                ).std()
            )
        )

        df_panel[f"{col}_rollstd5"] = (
            df_panel.groupby("COUNTRY")[col]
            .transform(
                lambda x: x.shift(1).rolling(
                    window=5,
                    min_periods=3,
                ).std()
            )
        )

print("Volatility features added.")

Volatility features added.


In [24]:
stationary_cols  = ["GDP_Growth","Inflation","Exports","Imports",
                    "Fiscal_Balance","Current_Account"]
differenced_cols = ["Debt_diff","Expenditure_diff","Revenue_diff",
                    "Savings_diff","Investment_diff"]
all_feature_cols = stationary_cols + differenced_cols

for col in all_feature_cols:
    df_panel[f"{col}_lag1"] = df_panel.groupby("COUNTRY")[col]\
        .transform(lambda x: x.shift(1))

# Autoregressive features
df_panel["GDP_Growth_lag1"]      = df_panel.groupby("COUNTRY")["GDP_Growth"]\
    .transform(lambda x: x.shift(1))
df_panel["GDP_Growth_rollmean3"] = df_panel.groupby("COUNTRY")["GDP_Growth"]\
    .transform(lambda x: x.shift(1).rolling(3, min_periods=2).mean())
df_panel["Inflation_lag1_log"]   = np.sign(df_panel["Inflation_lag1"]) * \
    np.log1p(np.abs(df_panel["Inflation_lag1"]))

lag_feature_cols = [f"{col}_lag1" for col in all_feature_cols]
print(f"Lag features created: {len(lag_feature_cols)}")
print(f"Total columns: {df_panel.shape[1]}")

Lag features created: 11
Total columns: 84


In [25]:
# Collect newly created feature columns without duplicates
raw_new_cols = (
    lag_feature_cols
    + [
        col for col in df_panel.columns
        if "rollmean" in col or "rollstd" in col
    ]
    + [
        "GDP_Growth_lag1",
        "GDP_Growth_rollmean3",
        "Inflation_lag1_log",
    ]
)

new_cols = list(dict.fromkeys(raw_new_cols))

# Lagged and rolling features must not be backward-filled because
# that would give earlier observations information from future years.
lag_and_rolling_cols = [
    col for col in new_cols
    if "lag" in col or "roll" in col
]

# Keep only rows with a known target and complete predictive history
df_panel = df_panel.dropna(
    subset=lag_and_rolling_cols + ["GDP_Growth"]
)

print(f"Final shape : {df_panel.shape}")
print(f"Countries   : {df_panel['COUNTRY'].nunique()}")
print(
    "Feature NaN:",
    df_panel[lag_and_rolling_cols].isna().sum().sum()
)

Final shape : (4982, 84)
Countries   : 175
Feature NaN: 0


In [26]:
# ── Save checkpoint ───────────────────────────────────────────
df_panel.to_csv("data/02_panel_features.csv", index=False)
print("Saved: data/02_panel_features.csv")
print(df_panel.shape)

Saved: data/02_panel_features.csv
(4982, 84)


In [27]:
df_panel.head()

,COUNTRY,YEAR,Inflation,Current_Account,Expenditure,Exports,Investment,Debt,GDP_Growth,Savings,...,Exports_lag1,Imports_lag1,Fiscal_Balance_lag1,Current_Account_lag1,Debt_diff_lag1,Expenditure_diff_lag1,Revenue_diff_lag1,Savings_diff_lag1,Investment_diff_lag1,Inflation_lag1_log
3,"Afghanistan, Islamic Republic of",2006,6.785,20.844,18.262,-5.943,29.489,22.985,5.361,50.333,...,40.082,54.919,-0.917,30.226,-38.611,0.582,2.057,-5.296,1.694,2.448329
4,"Afghanistan, Islamic Republic of",2007,8.681,63.390,21.448,-12.758,55.852,20.137,13.340,119.243,...,-5.943,-2.315,0.684,20.844,-183.371,2.611,4.213,-16.941,-7.559,2.052199
5,"Afghanistan, Islamic Republic of",2008,26.419,33.769,20.899,-5.885,30.222,19.057,3.863,63.990,...,-12.758,-10.316,-2.462,63.390,-2.848,3.186,0.040,68.910,26.363,2.270165
6,"Afghanistan, Islamic Republic of",2009,-6.811,41.587,21.151,35.076,36.503,16.247,20.585,78.090,...,-5.885,-8.349,-3.864,33.769,-1.080,-0.549,-1.950,-55.253,-25.630,3.311236
7,"Afghanistan, Islamic Republic of",2010,2.179,29.430,20.789,7.125,30.269,7.697,8.438,59.699,...,35.076,30.292,-1.761,41.587,-2.810,0.252,2.355,14.100,6.281,-2.055533
